# 27 · Canvas app fundamentals

## Goal

Bind the environment to Power Platform Git Integration, build the browse
(gallery) and detail (form) screens for the Renewal Desk app in the
designer, and see the sync land as reviewable YAML in
`apps/renewal-desk-canvas/src`.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../apps/renewal-desk-canvas").exists(), "app workspace missing — check repo layout"


## Concept

**Finding (Part 2, doc-checked 17 Aug 2026): `pac canvas pack`/`unpack` is
deprecated.** Microsoft's supported path for canvas app source control is
now **Power Platform Git Integration (GA)** — bind an environment to a
repo (Azure DevOps or GitHub), and every save in the designer unpacks the
`.msapp` into per-screen YAML/JSON that syncs back automatically. That's
the mechanism this notebook uses, not the deprecated CLI pair — `pac
canvas pack`/`unpack` still work and `csx.pac.canvas_pack`/`canvas_unpack`
wrap them for the narrow case of scripted CI-side diffing, but they are
not where you should point a new build.

The screens themselves are still built in the WYSIWYG designer — there is
no credible "author a canvas app screen by hand-writing YAML" path today.
That's the checkpoint below, same pattern as `05`'s SharePoint consent
step: a portal action the notebook verifies happened, rather than
pretending it can be scripted away.


## Build


### Bind Git Integration


In [ ]:
from csx.checkpoint import checkpoint
checkpoint(
    name="Environment bound to Power Platform Git Integration, pointed at this repo",
    probe=lambda: input("Bound in the maker portal (Solutions > this app > connect to Git), pointed at this repo/branch? (y/n): ") == "y",
    remediation="In the maker portal: open the environment > Settings > Git Integration (or the app's own Connect to Git action) > authenticate > select this repo and a working branch.",
)


### Build the two screens (designer)


In the canvas app designer: create **SupplierBrowse** (a gallery bound
to `crd_supplierrenewals`, sorted by `crd_contractenddate`) and
**SupplierDetail** (a form bound to the gallery's `Selected` item). Save.
Git Integration syncs both as YAML under `apps/renewal-desk-canvas/src/`.


In [ ]:
from csx.checkpoint import checkpoint
from pathlib import Path

def probe_synced_screens():
    src = Path("../apps/renewal-desk-canvas/src/Screens")
    return src.exists() and (src / "SupplierBrowse.pa.yaml").exists() and (src / "SupplierDetail.pa.yaml").exists()

checkpoint(
    name="Both screens synced to apps/renewal-desk-canvas/src via Git Integration",
    probe=probe_synced_screens,
    remediation="Save the app in the designer again — Git Integration syncs on save, not continuously. Then git pull inside the environment's connected repo view if needed.",
)


### Read the synced source — this is what you'll diff and review from here on


In [ ]:
from pathlib import Path
print((Path("../apps/renewal-desk-canvas/src/Screens/SupplierBrowse.pa.yaml")).read_text())


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import yaml
from pathlib import Path
browse = yaml.safe_load((Path("../apps/renewal-desk-canvas/src/Screens/SupplierBrowse.pa.yaml")).read_text())
detail = yaml.safe_load((Path("../apps/renewal-desk-canvas/src/Screens/SupplierDetail.pa.yaml")).read_text())
assert "SupplierBrowse" in browse.get("Screens", {}), "SupplierBrowse screen not found in synced source"
assert "SupplierDetail" in detail.get("Screens", {}), "SupplierDetail screen not found in synced source"
print("both screens present in synced source — structurally what a reviewer would diff on a PR")


## Cost


In [ ]:
print("Canvas app authoring/testing in the designer does not consume Copilot Credits — that meter is specific to the GHCP agent harness, not Power Apps.")


## Teardown


In [ ]:
print("No teardown — the two screens persist; 28 adds the agent chat control to SupplierDetail.")
